# Buổi G3: Chọn cột, bỏ cột thừa mà không tự lừa mình

**Hiện trường.** Bảng dữ liệu của chị Hạnh có hàng trăm cột: đủ loại chỉ số, nhiều cột không liên quan gì đến thứ cần đoán. Chị hỏi: *giữ hết hay bỏ bớt? bỏ cột nào? và làm sao biết mình bỏ đúng?*

Buổi này dạy ba nhóm cách **chọn cột** (*feature selection*):
- **Bộ lọc** (*filter*): chấm điểm từng cột riêng lẻ (tương quan, thông tin tương hỗ), giữ cột điểm cao.
- **Bọc** (*wrapper*): thử bỏ dần cột, mỗi lần học lại mô hình (RFE).
- **Nhúng** (*embedded*): để mô hình tự bỏ cột trong lúc học (Lasso, độ quan trọng của cây).

Và một cái bẫy lớn: **chọn cột bằng cả bộ dữ liệu rồi mới kiểm định chéo** là rò rỉ (D3), có thể làm một bài toán vô vọng trông như giải được.

**Ẩn dụ:** soạn hành lý đi du lịch. Bộ lọc là nhìn từng món và hỏi "món này có hay dùng không". Bọc là thử xách vali, bỏ từng món, xem có nhẹ mà vẫn đủ không. Nhúng là để chiếc vali tự nặng nhẹ khi ta xếp đồ.

**Mindset của người làm dữ liệu:** bỏ cột phải **nằm trong pipeline**, được học từ tập học, như mọi bước chế biến khác.

**Lưu ý:** dùng `load_breast_cancer` (30 cột thật) cộng cột nhiễu ngẫu nhiên **dựng có chủ đích** để biết đáp án.

## Bản đồ các chặng

| Chặng | Câu hỏi của chặng |
|---|---|
| Chặng 0 | Thấy ngay: thêm 100 cột nhiễu, mô hình tệ đi bao nhiêu |
| Chặng 1 | Bộ lọc: chấm điểm từng cột |
| Chặng 2 | Bọc: bỏ dần bằng RFE |
| Chặng 3 | Nhúng: Lasso và cây tự bỏ cột |
| Chặng 4 | Bẫy: chọn cột ngoài kiểm định chéo |
| Chặng 5 | Chọn số cột và độ ổn định của danh sách |

## Chặng 0: Thấy ngay, thêm cột nhiễu

**Câu hỏi:** thêm 100 cột vô nghĩa thì mô hình tệ đi bao nhiêu?

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
BLUE, GRAY = "#1F4E79", "#8C8C8C"
plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

**Giải thích từng dòng**

- `import numpy as np`: mượn thư viện numpy (máy tính bỏ túi cho cả dãy số), đặt tên ngắn là `np` để gõ cho nhanh.
- `import pandas as pd`: mượn pandas, thư viện làm việc với bảng số liệu giống Excel, tên ngắn là `pd`.
- `import matplotlib.pyplot as plt`: mượn công cụ vẽ biểu đồ, tên ngắn là `plt`.
- `BLUE, GRAY = "#1F4E79", "#8C8C8C"`: gán hai màu dùng suốt bài (xanh đậm và xám). Một dòng gán cùng lúc hai biến, mỗi biến một mã màu.
- `plt.rcParams["figure.figsize"] = (6, 4)`: đặt cỡ hình mặc định: rộng 6, cao 4 (đơn vị inch).
- `plt.rcParams["axes.spines.top"] = False`: bỏ đường viền phía trên khung hình cho gọn mắt.
- `plt.rcParams["axes.spines.right"] = False`: bỏ luôn đường viền bên phải.

In [2]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
data = load_breast_cancer(as_frame=True)
X_real, y = data.data, data.target
rng = np.random.RandomState(0)
noise = pd.DataFrame(rng.randn(len(X_real), 100), columns=[f"noise_{i}" for i in range(100)], index=X_real.index)
X_all = pd.concat([X_real, noise], axis=1)
cv = StratifiedKFold(5, shuffle=True, random_state=0)
models = {
    "logistic": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    "5-NN": make_pipeline(StandardScaler(), KNeighborsClassifier(5)),
    "forest": RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1),
}
rows = {}
for name, mdl in models.items():
    rows[name] = {"30 real columns": cross_val_score(mdl, X_real, y, cv=cv).mean(), "30 real + 100 noise": cross_val_score(mdl, X_all, y, cv=cv).mean()}
noise_table = pd.DataFrame(rows).T.round(4)
noise_table["drop (points)"] = ((noise_table["30 real columns"] - noise_table["30 real + 100 noise"]) * 100).round(2)
print(noise_table.to_string())

          30 real columns  30 real + 100 noise  drop (points)
logistic           0.9789               0.9491           2.98
5-NN               0.9649               0.8981           6.68
forest             0.9667               0.9526           1.41


**Giải thích từng dòng**

- `load_breast_cancer(as_frame=True)`: 569 ca u vú, 30 chỉ số đo từ ảnh tế bào, nhãn lành tính hoặc ác tính (C2). Dữ liệu mẫu để học.
- `rng.randn(len(X_real), 100)`: 100 cột **số ngẫu nhiên thuần túy**, không liên quan gì đến nhãn. Dựng có chủ đích.
- `pd.concat([X_real, noise], axis=1)`: ghép cột thật và cột nhiễu thành bảng 130 cột.
- `cross_val_score(mdl, X, y, cv=cv)`: kiểm định chéo 5 vòng (D1); lấy điểm trung bình.
- `"drop (points)"`: điểm mất đi (tính theo điểm phần trăm) khi có thêm cột nhiễu.

**Cách đọc.**
- **Con số:** điểm mất khi thêm 100 cột nhiễu: logistic **2,98** điểm, 5-NN **6,68** điểm, rừng **1,41** điểm.
- **Bức tranh:** mô hình nào nhạy với cột thừa thì tụt nhiều (k-NN dùng khoảng cách, cột nhiễu làm các điểm "xa nhau" như nhau); mô hình nào có cơ chế tự bỏ qua (rừng, khi cột nhiễu không giúp chia) thì tụt ít. Không phải mô hình nào cũng cần chọn cột, nhưng cột thừa không bao giờ làm tốt hơn.
- **Câu hỏi tiếp:** làm sao nhận ra cột thừa mà không biết trước đáp án?

**Mang về chặng 1:** cột thừa làm hại những mô hình dựa vào khoảng cách; chọn cột giúp gọn và đôi khi chính xác hơn.

## Chặng 1: Bộ lọc, chấm điểm từng cột

**Câu hỏi:** chấm điểm từng cột theo mức liên quan với nhãn, rồi xem cột nhiễu rơi xuống đâu?

In [3]:
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
f_scores, f_p = f_classif(X_all, y)
mi = mutual_info_classif(X_all, y, random_state=0)
rank = pd.DataFrame({"F score": f_scores, "mutual info": mi}, index=X_all.columns)
rank["is_noise"] = rank.index.str.startswith("noise")
for col in ["F score", "mutual info"]:
    top30 = rank.sort_values(col, ascending=False).head(30)
    print(f"{col}: real columns in the top 30 =", int((~top30["is_noise"]).sum()), "of 30")
print("highest-scoring noise column by F:", rank[rank["is_noise"]]["F score"].max().round(2), "| lowest real column by F:", rank[~rank["is_noise"]]["F score"].min().round(2))

F score: real columns in the top 30 = 25 of 30
mutual info: real columns in the top 30 = 26 of 30
highest-scoring noise column by F: 7.9 | lowest real column by F: 0.02


**Giải thích từng dòng**

- `f_classif(X_all, y)`: kiểm định thống kê **ANOVA F** cho từng cột: nhóm ác tính và lành tính có trung bình cột này khác nhau nhiều không. F càng cao càng liên quan.
- `mutual_info_classif(X_all, y)`: **thông tin tương hỗ**: biết giá trị cột này thì bớt bất định về nhãn bao nhiêu. Bắt được cả quan hệ cong, không chỉ tuyến tính.
- `rank.sort_values(col).head(30)`: lấy 30 cột điểm cao nhất, đếm bao nhiêu là cột thật.

**Cách đọc.**
- **Con số:** trong 30 cột điểm cao nhất theo F có **25** cột thật; theo thông tin tương hỗ **26**. Cột nhiễu cao điểm nhất có F **7,90**, cột thật thấp điểm nhất có F **0,02**.
- **Bức tranh:** bộ lọc xếp phần lớn cột thật lên đầu và dồn cột nhiễu xuống cuối; một vài cột thật điểm thấp bị lẫn với nhiễu. Bộ lọc nhanh vì chỉ nhìn **từng cột một**, nhưng cũng vì vậy nó **không thấy** tổ hợp (hai cột riêng lẻ yếu nhưng cùng nhau mạnh) và **không thấy trùng lặp** (hai cột giống hệt cùng được điểm cao).
- **Câu hỏi tiếp:** có cách nào nhìn cả tổ hợp?

**Mang về chặng 2:** bộ lọc nhanh, đơn giản, nhưng nhìn từng cột riêng lẻ.

## Chặng 2: Bọc, bỏ dần bằng RFE

**Câu hỏi:** học mô hình, bỏ cột yếu nhất, học lại, lặp lại: cuối cùng còn những cột nào?

In [4]:
from sklearn.feature_selection import RFE
import time
Xs = StandardScaler().fit_transform(X_all)
t0 = time.time()
rfe = RFE(LogisticRegression(max_iter=2000), n_features_to_select=20, step=5).fit(Xs, y)
t_rfe = round(time.time() - t0, 1)
kept = pd.Series(rfe.support_, index=X_all.columns)
print("kept 20 columns -> real:", int((kept & ~rank["is_noise"]).sum()), "| noise:", int((kept & rank["is_noise"]).sum()), "| seconds:", t_rfe)
print("kept real columns (first 8):", list(kept[kept & ~rank["is_noise"]].index[:8]))

kept 20 columns -> real: 13 | noise: 7 | seconds: 0.1
kept real columns (first 8): ['mean concavity', 'mean concave points', 'radius error', 'perimeter error', 'area error', 'fractal dimension error', 'worst radius', 'worst texture']


**Giải thích từng dòng**

- `RFE(LogisticRegression(), n_features_to_select=20, step=5)`: **loại bỏ đệ quy** (*recursive feature elimination*): học logistic, bỏ 5 cột có hệ số nhỏ nhất, học lại, lặp đến khi còn 20 cột.
- `StandardScaler().fit_transform(X_all)`: chuẩn hóa để hệ số các cột so được với nhau (G1, F1).
- `rfe.support_`: mảng đúng/sai cho từng cột: có được giữ không.

**Cách đọc.**
- **Con số:** RFE giữ 20 cột trong đó **13** cột thật và **7** cột nhiễu; chạy mất **0,1** giây.
- **Bức tranh:** vì mỗi vòng học lại mô hình trên **tất cả cột còn lại cùng lúc**, RFE thấy được quan hệ giữa các cột (hai cột trùng nhau thì cột kia hệ số nhỏ đi), mà bộ lọc không thấy. Cái giá là chậm hơn bộ lọc nhiều, và kết quả phụ thuộc mô hình dùng để bỏ.
- **Câu hỏi tiếp:** có cách nào mô hình tự bỏ cột ngay trong lúc học?

**Mang về chặng 3:** bọc chọn cột bằng chính mô hình, chính xác hơn nhưng tốn thời gian hơn.

## Chặng 3: Nhúng, Lasso và cây tự bỏ cột

**Câu hỏi:** để mô hình tự đẩy hệ số cột thừa về 0?

In [5]:
from sklearn.feature_selection import SelectFromModel
lasso = LogisticRegression(l1_ratio=1.0, solver="liblinear", C=0.1).fit(Xs, y)
l1_kept = pd.Series(lasso.coef_[0] != 0, index=X_all.columns)
forest = RandomForestClassifier(n_estimators=300, random_state=0, n_jobs=-1).fit(X_all, y)
sel_tree = SelectFromModel(forest, threshold="median", prefit=True)
tree_kept = pd.Series(sel_tree.get_support(), index=X_all.columns)
for name, k in [("L1 logistic (C=0.1)", l1_kept), ("forest importance >= median", tree_kept)]:
    print(f"{name}: kept", int(k.sum()), "| real", int((k & ~rank['is_noise']).sum()), "| noise", int((k & rank['is_noise']).sum()))

L1 logistic (C=0.1): kept 11 | real 8 | noise 3
forest importance >= median: kept 65 | real 30 | noise 35


**Giải thích từng dòng**

- `LogisticRegression(l1_ratio=1.0, solver="liblinear", C=0.1)`: hồi quy logistic có phạt **L1** (`l1_ratio=1.0` nghĩa là phạt L1 thuần; như Lasso ở B6): ép hệ số cột ít giá trị về **đúng 0**. `C` nhỏ thì phạt mạnh, bỏ nhiều.
- `lasso.coef_[0] != 0`: cột có hệ số khác 0 là cột được giữ.
- `SelectFromModel(forest, threshold="median")`: giữ cột có độ quan trọng của rừng từ mức trung vị trở lên (một nửa cột).

**Cách đọc.**
- **Con số:** L1 giữ **11** cột (thật **8**, nhiễu **3**); rừng giữ **65** cột (thật **30**, nhiễu **35**).
- **Bức tranh:** L1 chủ động đưa hệ số về 0 nên danh sách gọn và đọc được; ngưỡng "trung vị" của rừng **luôn giữ một nửa số cột** (65 trên 130) nên chắc chắn lọt nhiều cột nhiễu; độ quan trọng của rừng cũng đẩy cột nhiễu xuống không triệt để (như đã thấy ở E2, E6). Số cột giữ phụ thuộc vào mức phạt `C` hoặc ngưỡng, không có con số "đúng".
- **Quy tắc đọc:** hai bộ chọn cho hai danh sách khác nhau; phần **giao** của chúng đáng tin hơn cả hai.
- **Câu hỏi tiếp:** các cách này dùng cả bộ dữ liệu để chọn. Có sao không?

**Mang về chặng 4:** nhúng chọn cột ngay trong lúc học; số cột giữ phụ thuộc núm vặn.

## Chặng 4: Bẫy, chọn cột ngoài kiểm định chéo

**Câu hỏi:** nếu chọn cột bằng cả bộ dữ liệu rồi mới kiểm định chéo thì kết quả có đáng tin?

In [6]:
rng2 = np.random.RandomState(1)
Xr = rng2.randn(120, 2000)
yr = rng2.randint(0, 2, 120)
cv_r = StratifiedKFold(5, shuffle=True, random_state=0)
sel_all = SelectKBest(f_classif, k=20).fit(Xr, yr)
leaky = cross_val_score(LogisticRegression(max_iter=2000), sel_all.transform(Xr), yr, cv=cv_r).mean()
proper = cross_val_score(make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression(max_iter=2000)), Xr, yr, cv=cv_r).mean()
print("data: 120 rows x 2000 random columns, random labels (no signal at all)")
print("select 20 columns on ALL rows, then cross-validate:", round(leaky, 3))
print("select 20 columns INSIDE each fold (pipeline)     :", round(proper, 3))

data: 120 rows x 2000 random columns, random labels (no signal at all)
select 20 columns on ALL rows, then cross-validate: 0.825
select 20 columns INSIDE each fold (pipeline)     : 0.492


**Giải thích từng dòng**

- `rng2.randn(120, 2000)`: 120 dòng, 2000 cột số ngẫu nhiên; `yr` là nhãn 0 hoặc 1 tung đồng xu. **Không có bất kỳ tín hiệu nào** để học: điểm đúng chỉ có thể quanh 50%.
- `SelectKBest(f_classif, k=20).fit(Xr, yr)`: chọn 20 cột "liên quan nhất" bằng **toàn bộ** dòng, kể cả những dòng sau này làm tập thi. Đây là rò rỉ.
- `make_pipeline(SelectKBest(...), LogisticRegression())`: cách đúng: bước chọn cột nằm trong pipeline, nên mỗi vòng kiểm định chéo chọn lại **chỉ từ phần học** (D3).

**Cách đọc.**
- **Con số:** chọn cột bằng toàn bộ dữ liệu rồi kiểm định chéo cho **82,5%**; đặt bước chọn cột trong pipeline cho **49,2%**.
- **Bức tranh:** trong 2000 cột ngẫu nhiên, luôn có vài cột tình cờ trùng với nhãn. Khi chọn bằng cả bộ dữ liệu, ta đã "nhìn trước" đáp án của các dòng sẽ thi, nên điểm cao ảo. Pipeline đúng cho điểm quanh 50%, đúng với sự thật là dữ liệu này không có gì để học.
- **Quy tắc đọc:** mọi bước **học từ dữ liệu** (chuẩn hóa, điền ô trống, **chọn cột**, PCA) đều phải nằm trong pipeline. Dữ liệu càng nhiều cột và càng ít dòng thì hậu quả của sai lầm này càng nặng.

**Mang về chặng 5:** chọn cột là một bước học; kiểm định chéo phải bao cả nó.

## Chặng 5: Chọn số cột và độ ổn định

**Câu hỏi:** giữ bao nhiêu cột, và danh sách cột được chọn có ổn định không?

In [7]:
rows = []
for k in [5, 10, 20, 30, 60, 130]:
    pipe = make_pipeline(StandardScaler(), SelectKBest(f_classif, k=k), LogisticRegression(max_iter=2000))
    s = cross_val_score(pipe, X_all, y, cv=cv)
    rows.append({"k columns": k, "accuracy": round(s.mean(), 4), "std": round(s.std(), 4)})
k_table = pd.DataFrame(rows).set_index("k columns")
print(k_table.to_string())
picks = []
for tr, _ in cv.split(X_all, y):
    sk = SelectKBest(f_classif, k=10).fit(X_all.iloc[tr], y.iloc[tr])
    picks.append(set(X_all.columns[sk.get_support()]))
always = set.intersection(*picks); ever = set.union(*picks)
print("top-10 list per fold: columns chosen in every fold =", len(always), "| chosen in at least one fold =", len(ever))

           accuracy     std
k columns                  
5            0.9473  0.0298
10           0.9596  0.0252
20           0.9842  0.0102
30           0.9719  0.0129
60           0.9561  0.0184
130          0.9491  0.0217
top-10 list per fold: columns chosen in every fold = 10 | chosen in at least one fold = 10


**Giải thích từng dòng**

- `SelectKBest(f_classif, k=k)`: giữ `k` cột điểm F cao nhất, **trong pipeline** nên chọn lại ở mỗi vòng.
- `cv.split(X_all, y)`: lấy từng phần học của 5 vòng kiểm định chéo.
- `set.intersection(*picks) / set.union(*picks)`: các cột có mặt ở **mọi** vòng và các cột có mặt ở **ít nhất một** vòng. Hai số này gần nhau thì danh sách ổn định.

**Cách đọc.**
- **Con số:** điểm kiểm định chéo theo số cột giữ: k=5 **94,7%**, k=10 **96,0%**, k=30 **97,2%**, k=130 (giữ hết) **94,9%**. Danh sách top 10 có **10** cột có mặt ở cả 5 vòng, và **10** cột xuất hiện ở ít nhất một vòng.
- **Bức tranh:** điểm đạt đỉnh quanh **k = 20** (98,4%) rồi giảm dần khi thêm cột; giữ hết 130 cột chỉ còn 94,9%, vì phần thêm vào chủ yếu là nhiễu. Các `k` gần nhau chênh trong khoảng dao động (độ lệch trung bình cỡ 2,0%), nên đừng đọc quá kỹ từng con số: chọn `k` nhỏ nhất mà điểm chưa tụt rõ, vì ít cột thì dễ giải thích và rẻ hơn.
- **Quy tắc đọc:** ở bài này danh sách top 10 ổn định (cột nào cũng có mặt ở cả 5 vòng) vì 30 cột thật có tín hiệu mạnh. Nếu danh sách đổi nhiều giữa các vòng thì đừng kể câu chuyện "cột A quan trọng nhất" từ một lần chọn; chỉ tin những cột có mặt ở mọi vòng.

**Mang về:** lọc cho nhanh, bọc cho chính xác, nhúng cho gọn; tất cả nằm trong pipeline; chọn `k` bằng kiểm định chéo; chỉ tin cột ổn định.

## Tổng kết

**Mang về theo từng chặng**

| Chặng | Mang về |
|---|---|
| 0 | Cột nhiễu hại những mô hình dựa vào khoảng cách; chọn cột giúp gọn và đôi khi chính xác. |
| 1 | Bộ lọc nhanh nhưng nhìn từng cột riêng lẻ. |
| 2 | RFE bỏ dần bằng chính mô hình: thấy quan hệ giữa cột, nhưng chậm. |
| 3 | Lasso và cây tự bỏ cột; số cột giữ phụ thuộc núm vặn. |
| 4 | Chọn cột ngoài kiểm định chéo là rò rỉ; đặt trong pipeline. |
| 5 | Chọn k bằng kiểm định chéo; chỉ tin cột ổn định qua các vòng. |

**Bảng đối chiếu: đời thường, thuật ngữ, tên trong code**

| Đời thường | Thuật ngữ | Tên trong code |
|---|---|---|
| Nhìn từng món xem có hay dùng không | bộ lọc | `SelectKBest`, `f_classif` |
| Thử bỏ từng món xem vali còn đủ không | bọc | `RFE` |
| Vali tự nặng nhẹ khi xếp đồ | nhúng | `SelectFromModel`, `l1_ratio=1.0` |
| Biết món này giúp bớt chắc về nhãn bao nhiêu | thông tin tương hỗ | `mutual_info_classif` |
| Cột có mặt ở mọi vòng | độ ổn định danh sách | `set.intersection` |
| Chọn cột nằm trong từng vòng thi | chọn cột trong pipeline | `make_pipeline(SelectKBest, model)` |

**Thử thách tự làm**

1. Ở chặng 0, thêm 400 cột nhiễu thay vì 100. Mô hình nào tụt thêm nhiều nhất?
2. Ở chặng 3, đổi `C` thành 1 và 0,01. Số cột L1 giữ thế nào? Có còn cột nhiễu nào lọt không?
3. Ở chặng 4, tăng số dòng từ 120 lên 1200 (giữ 2000 cột). Khoảng cách giữa cách sai và cách đúng thu hẹp ra sao?
4. Thay `f_classif` bằng `mutual_info_classif` ở chặng 5. Kết quả có khác không?

**Checklist năng lực** (tự tick khi làm được)

- [ ] Phân biệt bộ lọc, bọc, nhúng.
- [ ] Chọn cột trong pipeline, không ngoài.
- [ ] Chọn số cột giữ bằng kiểm định chéo.
- [ ] Kiểm tra độ ổn định của danh sách cột.
- [ ] Biết giới hạn của bộ lọc: không thấy tổ hợp và trùng lặp.

## Phần thưởng (Tip pro): Sang G4: tạo cột mới


Chọn cột là bớt đi. G4 làm ngược lại: **tạo cột mới** từ cột cũ (bình phương, tích hai cột, chia khoảng, giờ đổi thành sin và cos) để mô hình đơn giản như hồi quy tuyến tính cũng bắt được quan hệ cong.